In [ ]:
import csv
import os

In [ ]:
def load_ancient_ids(ancient_ids_file):
    """
    Load the ancient IDs file and return a list of SampleName values.
    The file is assumed to be a TSV with a header including "SampleName".
    """
    sample_names = []
    with open(ancient_ids_file, 'r', newline='') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            sample_names.append(row['SampleName'])
    return sample_names

def filter_tsv(tsv_file, ancient_sample_names, output_dir):
    """
    Filter a TSV file by checking if the query (first column) is found as a substring in any
    of the ancient SampleName entries.
    Writes out the filtered TSV (keeping the header) into output_dir with the same filename.
    """
    basename = os.path.basename(tsv_file)
    output_file = os.path.join(output_dir, basename)
    with open(tsv_file, 'r', newline='') as inf, open(output_file, 'w', newline='') as outf:
        reader = csv.reader(inf, delimiter='\t')
        writer = csv.writer(outf, delimiter='\t')
        header = next(reader)
        writer.writerow(header)
        for row in reader:
            if not row:
                continue
            query = row[0]
            # Check if query is a substring of any SampleName
            if any(query in sample for sample in ancient_sample_names):
                writer.writerow(row)
    print(f"Filtered {tsv_file}: output saved to {output_file}")

In [ ]:
# Set your paths directly:
ancient_ids_file = "/project3/data/results/other-evals/ancientness/ancient_orfs_min100.tsv"  # Path to the ancient IDs file (TSV)
tsv_dir = "/project3/data/results/assembly-prokka-mmseqs-nr/merged/grouped/pathways"                        # Directory containing TSV files to filter
result_dir = "/project3/data/results/assembly-prokka-mmseqs-nr/merged/grouped/pathways/ancientMin100"                               # Output directory for filtered TSV files

# Create result directory if it doesn't exist.
if not os.path.exists(result_dir):
    os.makedirs(result_dir)

# Load ancient sample names
ancient_sample_names = load_ancient_ids(ancient_ids_file)

# Process each TSV file in the specified directory.
for f in os.listdir(tsv_dir):
    if f.endswith('.tsv'):
        tsv_file = os.path.join(tsv_dir, f)
        filter_tsv(tsv_file, ancient_sample_names, result_dir)
